In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
import os
import pickle
import logging
import warnings
import re
from typing import Dict, List
from pathlib import Path
import json 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.gridspec as gridspec
import geopandas as gpd

import torch
import massbalancemachine as mbm

import math
import string
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import root_mean_squared_error

from regions.RGI_11_Switzerland.scripts.plotting.palettes import get_color_maps
from regions.RGI_11_Switzerland.scripts.geo_data import apply_gaussian_filter

# reuse _load_lstm_ds from the existing plotting module — reads {rgi_id}/{rgi_id}_{year}_{period}.zarr
from regions.RGI_11_Switzerland.scripts.plotting.spatial_maps import _load_lstm_ds

sys.path.append(os.path.join(os.getcwd(), '../../'))
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.config_models import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"FD_MODEL"
CACHE_DIR = BASE_DIR / "datasets/"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

BARPLOT_ALPHA = 0.7

In [ ]:
def to_roman(n):
    vals = [(1000, 'm'), (900, 'cm'), (500, 'd'), (400, 'cd'), (100, 'c'),
            (90, 'xc'), (50, 'l'), (40, 'xl'), (10, 'x'), (9, 'ix'), (5, 'v'),
            (4, 'iv'), (1, 'i')]
    result = ''
    for v, s in vals:
        while n >= v:
            result += s
            n -= v
    return result

## Datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
print(num_measurements)

SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
    'IT_AT': False
}
# Compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

### Subregions:

In [ ]:
# Central Asia
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)

# Map O2Region into monthly_cache for CENTRALASIA
monthly_cache['CENTRALASIA']['data_monthly']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))
monthly_cache['CENTRALASIA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly_aug']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))

# Central Asia subregions (O2Region already mapped onto dfs['13'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

# Alaska
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)

# Map O2Region into monthly_cache for ALA
monthly_cache['ALA']['data_monthly']['O2Region'] = (
    monthly_cache['ALA']['data_monthly']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))
monthly_cache['ALA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['ALA']['data_monthly_aug']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))

# Alaska subregions (O2Region already mapped onto dfs['01'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

# IT/AT split
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

# With subregions
TARGET_REGIONS_SUB = [
    'IT', 'AT', 'SJM', 'ALA', 'CA_3', 'CA_4'
    #'CENTRALASIA', 'ALA', #'CA_12',
]

XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

### Pooled set:

In [ ]:
# Build glacier→region mapping from all source regions (no ranking needed)
df_all_glaciers = pd.concat(
    [
        res_xreg_by_source[region]["df_train"][[
            "GLACIER"
        ]].drop_duplicates().assign(region=region) for region in SOURCE_REGIONS
    ],
    ignore_index=True,
).rename(columns={"GLACIER": "glacier"})

# Add measurement counts (needed by build_assets_from_glacier_list internally? No — only used by build_glacier_subsets)
# df_ranked just needs columns: "glacier", "region"  ← that's all build_assets uses
glaciers_all = df_all_glaciers["glacier"].tolist()
print(f"Total unique glaciers across all source regions: {len(glaciers_all)}")

assets_full = build_assets_from_glacier_list(
    glaciers=glaciers_all,
    df_ranked=df_all_glaciers,  # acts as the gl→region lookup
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_path=str(CACHE_DIR / "assets_foundation_100pct.joblib"),
    force_recompute=False,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    src_regions=SOURCE_REGIONS,
)

# Failsafe: check that pooled training data only contains source region codes
actual_codes = set(df_all_glaciers.region.unique())
expected_codes = set(SOURCE_REGIONS)
unexpected = actual_codes - expected_codes
if unexpected:
    raise ValueError(
        f"Pooled training set contains unexpected SOURCE_CODEs: {unexpected}")
else:
    print(f"✓ Pooled set SOURCE_CODEs OK: {actual_codes}")

print(f"Pooled model assets: {len(assets_full['ds_train'])} sequences")

# ── Always compute scalers/blurs — needed for any recomputation ───────────────
res_source_only = {
    region: {
        "df_train": res_xreg_by_source[region]["df_train"],
    }
    for region in SOURCE_REGIONS
}

scaler_m, scaler_s, scaler_joint, scaler_geo = build_global_scalers_multi_source_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)

blur_m, blur_s, blur_joint, blur_geo = estimate_global_bandwidths_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
print(
    f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
)

### Hold-out set:

In [ ]:
RECOMPUTE_SPLITS = False

FT_FRAC = 0.10
save_path = BASE_DIR / "glacier_splits_FD.json"

gl_level_split = []
id_level_split = ['IT', 'AT', 'SJM', 'ALA', 'CA_3', 'CA_4']

# ── Load existing glacier splits from disk ────────────────────────────────────
splits = {}
if not RECOMPUTE_SPLITS and save_path.exists():
    with open(save_path, "r") as f:
        splits = json.load(f)
    print(f"Loaded splits from: {save_path}")

# ── Only compute glacier splits for regions that need them ────────────────────
regions_needing_gl_split = [r for r in gl_level_split]
regions_to_compute = (regions_needing_gl_split if RECOMPUTE_SPLITS else
                      [r for r in regions_needing_gl_split if r not in splits])

for region in regions_to_compute:
    print(f"\n{'='*50}\nSplitting region: {region}")
    split = split_pool_holdout_sinkhorn(
        df_region=monthly_cache[region]["data_monthly"],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        blur_joint=blur_joint,
        pool_frac=0.2,
        n_restarts=50,
        seed=cfg.seed,
    )
    splits[region] = split
    print(f"  Pool glaciers    : {split['pool_glaciers']}")
    print(f"  Holdout glaciers : {split['holdout_glaciers']}")
    print(f"  Pool fraction    : {split['actual_pool_frac']:.1%}")
    print(f"  Sk(pool↔holdout) : {split['sinkhorn_pool_vs_holdout']:.4f}")

if regions_to_compute:
    with open(save_path, "w") as f:
        json.dump(splits, f, indent=2)
    print(f"\nSaved splits for: {regions_to_compute}")

In [ ]:
FORCE_RECOMPUTE = False
FORCE_RECOMPUTE_REGIONS = []

N_REPEATS = 5

# ── Build FT assets for all target subregions, repeated N_REPEATS times ──────
ft_assets_glacier = {}
ft_assets_id = {region: [] for region in TARGET_REGIONS_SUB}

for region in TARGET_REGIONS_SUB:
    print(f"\n{'='*50}\nBuilding FT assets for: {region}")

    df_monthly = monthly_cache[region]["data_monthly"]
    df_monthly_aug = monthly_cache[region]["data_monthly_aug"]
    head_pad = monthly_cache[region]["months_head_pad"]
    tail_pad = monthly_cache[region]["months_tail_pad"]
    force = FORCE_RECOMPUTE or (region in FORCE_RECOMPUTE_REGIONS)

    # 1. Glacier-level split (unchanged — deterministic by glacier, no repeats needed)
    if region in gl_level_split:
        if region in gl_level_split:
            pool_glaciers = splits[region]["pool_glaciers"]
            holdout_glaciers = splits[region]["holdout_glaciers"]
            exp_key = f"FD_to_{region}"

            ds_ft_gl = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key}_FT",
                df_loss=df_monthly[df_monthly["GLACIER"].isin(pool_glaciers)],
                df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                    pool_glaciers)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="ft",
                show_progress=False,
            )
            ds_test_gl = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key}_TEST",
                df_loss=df_monthly[df_monthly["GLACIER"].isin(
                    holdout_glaciers)],
                df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                    holdout_glaciers)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="test",
                show_progress=False,
            )
            ft_train_idx_gl, ft_val_idx_gl = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_gl), val_ratio=0.2, seed=cfg.seed)
            ft_assets_glacier[region] = {
                "ds_ft": ds_ft_gl,
                "ds_test": ds_test_gl,
                "ft_train_idx": ft_train_idx_gl,
                "ft_val_idx": ft_val_idx_gl,
            }
            print(
                f"  [glacier-split] ds_ft  : {len(ds_ft_gl)} seqs | ds_test: {len(ds_test_gl)} seqs"
            )

    # 2. ID-level split, repeated N_REPEATS times with different seeds
    if region in id_level_split:
        winter_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                     "winter"]["ID"].unique().tolist()
        annual_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                     "annual"]["ID"].unique().tolist()

        for rep in range(N_REPEATS):
            winter_ids = list(winter_ids_full)
            annual_ids = list(annual_ids_full)

            rep_seed = cfg.seed + rep
            rng = np.random.default_rng(rep_seed)
            rng.shuffle(winter_ids)
            rng.shuffle(annual_ids)

            n_ft_w = max(1, int(len(winter_ids) *
                                FT_FRAC)) if winter_ids else 0
            n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))
            ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
            test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

            # keep track of ft_ids per rep for the cross-rep check below
            if rep == 0:
                _ft_id_sets_for_check = []
            _ft_id_sets_for_check.append(set(ft_ids))

            exp_key_id = f"FD_to_{region}_IDsplit_rep{rep}"
            ds_ft_id = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key_id}_FT",
                df_loss=df_monthly[df_monthly["ID"].isin(ft_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(ft_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="ft",
                show_progress=False,
            )
            ds_test_id = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key_id}_TEST",
                df_loss=df_monthly[df_monthly["ID"].isin(test_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(test_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="test",
                show_progress=False,
            )
            ft_train_idx_id, ft_val_idx_id = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_id), val_ratio=0.2, seed=rep_seed)

            ft_assets_id[region].append({
                "rep": rep,
                "seed": rep_seed,
                "ds_ft": ds_ft_id,
                "ds_test": ds_test_id,
                "ft_train_idx": ft_train_idx_id,
                "ft_val_idx": ft_val_idx_id,
            })
            print(f"  [ID-split rep {rep}] ds_ft  : {len(ds_ft_id)} seqs | "
                  f"ds_test: {len(ds_test_id)} seqs")

            # ── sanity check after all reps for this region are built ────────────
        if region in id_level_split:
            n_reps = len(_ft_id_sets_for_check)
            identical_pairs = [
                (i, j) for i in range(n_reps) for j in range(i + 1, n_reps)
                if _ft_id_sets_for_check[i] == _ft_id_sets_for_check[j]
            ]
            if identical_pairs:
                print(
                    f"  WARNING [{region}]: identical FT id-sets for repeats {identical_pairs}"
                )
            else:
                overlaps = [
                    len(_ft_id_sets_for_check[i] & _ft_id_sets_for_check[j]) /
                    len(_ft_id_sets_for_check[i]) for i in range(n_reps)
                    for j in range(i + 1, n_reps)
                ]
                print(
                    f"  [check] {region}: {n_reps} repeats distinct (mean overlap {np.mean(overlaps):.1%})"
                )

# convenience dict: glacier-split where available, ID-split (list of reps) for the rest
ft_assets = {
    **ft_assets_glacier,
    **{
        r: ft_assets_id[r]
        for r in ft_assets_id if r not in ft_assets_glacier
    },
}

## Model assets:

## Transformer:

### Zero shot:

In [ ]:
TRAIN_FLAG = False
models_dir = BASE_DIR / "models" / "foundation"
os.makedirs(models_dir, exist_ok=True)

model_pooled, model_path, info = train_or_load_one_source_model(
    cfg=cfg,
    key="foundation_100pct",
    lstm_assets=assets_full,
    best_params=PARAMS_TRANSFORMER,
    device=device,
    models_dir=models_dir,
    prefix="transformer_foundation",
    train_flag=TRAIN_FLAG,
    force_retrain=False,
    epochs=N_EPOCHS,
    date=MODEL_DATE,
    model_type="transformer",
    verbose=True,
)

print(f"Pooled model saved to: {model_path}")

# Build scaler once outside the loop
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_full["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=assets_full["train_idx"],
    val_idx=assets_full["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

print(f"\n{'='*60}")
print("Zero-shot evaluation of pooled model on target regions")
print(f"{'='*60}")

# ── ID-split: list of reps per region, aggregate across reps ─────────────────
print(f"\n--- ID-split (mean ± std over {N_REPEATS} repeats) ---")
zs_id_metrics = {
}  # store per-region rep metrics for later reuse (e.g. plotting)

for region in id_level_split:
    rep_metrics = []
    for rep_data in ft_assets_id[region]:
        ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            rep_data["ds_test"])
        test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
        metrics, _ = model_pooled.evaluate_with_preds(device, test_dl,
                                                      ds_test_copy)
        rep_metrics.append(metrics)

    zs_id_metrics[region] = rep_metrics

    rmse_a = [m["RMSE_annual"] for m in rep_metrics]
    rmse_w = [m["RMSE_winter"] for m in rep_metrics]
    r2_a = [m["R2_annual"] for m in rep_metrics]
    r2_w = [m["R2_winter"] for m in rep_metrics]

    print(f"  {region:15s}  "
          f"RMSE_a={np.mean(rmse_a):.3f}±{np.std(rmse_a):.3f}  "
          f"RMSE_w={np.mean(rmse_w):.3f}±{np.std(rmse_w):.3f}  "
          f"R2_a={np.mean(r2_a):.3f}±{np.std(r2_a):.3f}  "
          f"R2_w={np.mean(r2_w):.3f}±{np.std(r2_w):.3f}")

### Fine tuned model:

In [ ]:
def train_or_load_ft_models_for_region(
    region_code,
    monthly_cache,
    model_pooled,
    models_dir_ft,
    strategy,
    n_repeats,
    model_date,
    cfg,
    device,
    best_params,
    ds_pooled_scaler,
    ft_frac=0.10,
    force_retrain=False,
):
    """
    Train (or load from disk if already trained) N_REPEATS fine-tuned models
    for a single region using ID-level splitting, same pattern as CEU.
    """
    if region_code not in monthly_cache:
        raise KeyError(f"{region_code} not in monthly_cache")

    df_monthly = monthly_cache[region_code]["data_monthly"]
    df_monthly_aug = monthly_cache[region_code]["data_monthly_aug"]
    head_pad = monthly_cache[region_code]["months_head_pad"]
    tail_pad = monthly_cache[region_code]["months_tail_pad"]

    winter_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                 "winter"]["ID"].unique().tolist()
    annual_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                 "annual"]["ID"].unique().tolist()

    print(
        f"  {region_code}: {df_monthly['GLACIER'].nunique()} glaciers | "
        f"winter IDs: {len(winter_ids_full)} | annual IDs: {len(annual_ids_full)}"
    )

    ft_models = []
    for rep in range(n_repeats):
        model_path = str(
            models_dir_ft /
            f"transformer_ft_{strategy}_{region_code}_IDsplit_rep{rep}_{model_date}.pt"
        )

        winter_ids = list(winter_ids_full)
        annual_ids = list(annual_ids_full)
        rep_seed = cfg.seed + rep
        rng = np.random.default_rng(rep_seed)
        rng.shuffle(winter_ids)
        rng.shuffle(annual_ids)

        n_ft_w = max(1, int(len(winter_ids) * ft_frac)) if winter_ids else 0
        n_ft_a = max(1, int(len(annual_ids) * ft_frac))
        ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]

        ds_ft = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"FD_to_{region_code}_IDsplit_rep{rep}_FT",
            df_loss=df_monthly[df_monthly["ID"].isin(ft_ids)],
            df_full=df_monthly_aug[df_monthly_aug["ID"].isin(ft_ids)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=False,
            kind="ft",
            show_progress=False,
        )

        print(f"  [rep {rep}] ds_ft: {len(ds_ft)} seqs | model: {model_path}")

        ft_model = finetune_transformer_on_target(
            cfg=cfg,
            model_pooled=model_pooled,
            ds_ft=ds_ft,
            ds_pooled_scaler=ds_pooled_scaler,
            device=device,
            best_params=best_params,
            model_filename=model_path,
            strategy=strategy,
            epochs=PARAMS_FT["epochs"],
            lr_factor=PARAMS_FT["lr_factor"],
            force_retrain=force_retrain,  # False = load if exists, train if not
        )
        ft_models.append((rep, ft_model))
        print(f"  [rep {rep}] done")

    return ft_models


# ── Train or load fine-tuned models for all non-CEU regions ──────────────────
RGI_TO_FT_REGION = {
    "06": "ISL",
    "07": "SJM",
    "08": "NOR",
}

STRATEGY_OTHER = "full"
ft_models_by_rgi = {}  # no longer pre-populated with CEU
models_dir_ft = BASE_DIR / "models" / "finetuned"

for rgi_id, ft_code in RGI_TO_FT_REGION.items():
    print(f"\n{'='*60}\nRGI {rgi_id} ({ft_code})")
    ft_models_by_rgi[rgi_id] = train_or_load_ft_models_for_region(
        region_code=ft_code,
        monthly_cache=monthly_cache,
        model_pooled=model_pooled,
        models_dir_ft=models_dir_ft,
        strategy=STRATEGY_OTHER,
        n_repeats=N_REPEATS,
        model_date=MODEL_DATE,
        cfg=cfg,
        device=device,
        best_params=PARAMS_TRANSFORMER,
        ds_pooled_scaler=ds_pooled_scaler,
        ft_frac=FT_FRAC,
        force_retrain=False,
    )
    print(f"  -> {len(ft_models_by_rgi[rgi_id])} models ready")
    
ft_models_ch = train_or_load_ft_models_for_region(
    region_code="CH",
    monthly_cache=monthly_cache,
    model_pooled=model_pooled,
    models_dir_ft=models_dir_ft,
    strategy=STRATEGY_OTHER,
    n_repeats=N_REPEATS,
    model_date=MODEL_DATE,
    cfg=cfg,
    device=device,
    best_params=PARAMS_TRANSFORMER,
    ds_pooled_scaler=ds_pooled_scaler,
    ft_frac=FT_FRAC,
    force_retrain=False,
)
print(f"  -> {len(ft_models_ch)} CH fine-tuned models ready")

### Single source models:

In [ ]:
XREG_MODELS_BASE = Path(cfg.dataPath) / path_cache / "CROSS_REGION/models"
TRAIN_FLAG = False

single_source_models = {}
single_source_assets = {}
single_source_scalers = {}

for src in SOURCE_REGIONS:
    ckpt = XREG_MODELS_BASE / src / f"transformer_xreg_{src}_{src}_{MODEL_DATE}.pt"
    os.makedirs(ckpt.parent, exist_ok=True)

    df_src = res_xreg_by_source[src]["df_train"][["GLACIER"]] \
               .drop_duplicates().assign(region=src) \
               .rename(columns={"GLACIER": "glacier"})
    assets_src = build_assets_from_glacier_list(
        glaciers=df_src["glacier"].tolist(),
        df_ranked=df_src,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(XREG_MODELS_BASE / src / f"assets_{src}_100pct.joblib"),
        force_recompute=False,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        src_regions=[src],
    )
    single_source_assets[src] = assets_src

    # build per-source scaler
    scaler_src = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_src["ds_train"])
    scaler_src.make_loaders(
        train_idx=assets_src["train_idx"],
        val_idx=assets_src["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    single_source_scalers[src] = scaler_src

    m = mbm.models.Transformer_MB.build_model_from_params(cfg,
                                                          PARAMS_TRANSFORMER,
                                                          device,
                                                          verbose=False)
    if ckpt.exists():
        state = torch.load(ckpt, map_location=device)
        m.load_state_dict(state)
        print(f"Loaded {src} model from {ckpt}")
    else:
        print(f"No checkpoint found for {src}, training from scratch...")
        m, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=src,
            lstm_assets=assets_src,
            best_params=PARAMS_TRANSFORMER,
            device=device,
            models_dir=XREG_MODELS_BASE / src,
            prefix="transformer_xreg",
            train_flag=TRAIN_FLAG,
            force_retrain=False,
            epochs=N_EPOCHS,
            date=MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        print(f"Trained {src} model, saved to {ckpt}")

    single_source_models[src] = m

single_source_models_lstm = {}

for src in SOURCE_REGIONS:
    ckpt_lstm = XREG_MODELS_BASE / src / f"lstm_xreg_{src}_{MODEL_DATE}.pt"

    if not ckpt_lstm.exists():
        print(f"  No LSTM checkpoint found for {src}: {ckpt_lstm}, skipping")
        continue

    m_lstm = mbm.models.LSTM_MB.build_model_from_params(cfg,
                                                        PARAMS_LSTM,
                                                        device,
                                                        verbose=False)
    state = torch.load(ckpt_lstm, map_location=device)
    m_lstm.load_state_dict(state)
    m_lstm.eval()
    single_source_models_lstm[src] = m_lstm
    print(f"  Loaded LSTM {src} from {ckpt_lstm}")

print(
    f"Loaded {len(single_source_models_lstm)} LSTM single-source models: {list(single_source_models_lstm)}"
)

## Make predictions:

In [ ]:
def predict_grid_from_cached_datasets(
    cfg,
    ds_cache_dir,
    model,
    ds_pooled_scaler,
    device,
    region_folder,
    path_save_glw,
    months_head_pad,
    months_tail_pad,
    batch_size=512,
    skip_existing=True,
    glacier_filter=None,
    period="annual",
):
    basepath = os.path.join(cfg.dataPath, "RGI_v6", region_folder)
    zarr_dir = os.path.join(basepath, "xr_masked_grids")
    monthly_parquet_dir = os.path.join(basepath, "monthly_parquets")
    os.makedirs(path_save_glw, exist_ok=True)

    ds_paths = sorted(
        glob.glob(os.path.join(ds_cache_dir, "*", "*", "ds.joblib")))
    if not ds_paths:
        raise FileNotFoundError(
            f"No cached datasets found under {ds_cache_dir}")

    if glacier_filter is not None:
        glacier_filter = set(glacier_filter)
        ds_paths = [
            p for p in ds_paths if os.path.basename(
                os.path.dirname(os.path.dirname(p))) in glacier_filter
        ]

    print(f"{len(ds_paths)} cached (glacier, year) datasets to predict")

    n_rows_total = 0
    written_paths = []
    skipped_empty = 0

    for ds_path in tqdm(ds_paths, desc="Predicting per glacier-year"):
        rgi_id = os.path.basename(os.path.dirname(os.path.dirname(ds_path)))
        year = int(os.path.basename(os.path.dirname(ds_path)))

        out_zarr = os.path.join(path_save_glw, rgi_id,
                                f"{rgi_id}_{year}_{period}.zarr")
        if skip_existing and os.path.exists(out_zarr):
            written_paths.append(out_zarr)
            continue

        # load the masked WGS84 zarr — already has lat/lon dimensions
        xr_zarr_path = os.path.join(zarr_dir, f"{rgi_id}.zarr")
        if not os.path.exists(xr_zarr_path):
            print(
                f"  SKIP {rgi_id} {year}: masked zarr not found at {xr_zarr_path}"
            )
            continue
        ds_latlon = xr.open_zarr(xr_zarr_path)

        # load monthly parquet for spatial coordinates
        monthly_path = os.path.join(monthly_parquet_dir, rgi_id,
                                    f"{rgi_id}_grid_{year}.parquet")
        if not os.path.exists(monthly_path):
            print(f"  SKIP {rgi_id} {year}: monthly parquet not found")
            continue

        ds = joblib.load(ds_path)
        if len(ds) == 0:
            skipped_empty += 1
            continue

        ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            ds)
        dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_copy, ds_pooled_scaler, batch_size=batch_size, seed=cfg.seed)

        df_preds = model.predict_with_keys(device, dl, ds_copy, denorm=True)
        df_preds = df_preds[df_preds["PERIOD"] == period]

        if len(df_preds) == 0:
            skipped_empty += 1
            continue

        # merge with spatial coordinates from monthly parquet
        df_monthly = pd.read_parquet(monthly_path,
                                     columns=["ID", "POINT_LAT", "POINT_LON"
                                              ]).drop_duplicates(subset=["ID"])

        df_merged = df_preds[["ID", "pred"]].merge(df_monthly,
                                                   on="ID",
                                                   how="left")
        df_merged = df_merged.dropna(subset=["POINT_LAT", "POINT_LON", "pred"])

        if len(df_merged) == 0:
            skipped_empty += 1
            continue

        # save via GeoData._save_prediction — ds_latlon already has lat/lon dims
        geoData = mbm.geodata.GeoData(
            df_merged,
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
        )
        geoData._save_prediction(ds_latlon, df_merged, rgi_id, year,
                                 path_save_glw, period)

        n_rows_total += len(df_merged)
        written_paths.append(out_zarr)

        del ds, ds_copy, dl, df_preds, df_merged, ds_latlon
        gc.collect()

    print(
        f"Done. Wrote {n_rows_total} rows across {len(written_paths)} zarr files -> {path_save_glw}"
    )
    if skipped_empty:
        print(f"  ({skipped_empty} skipped: empty or missing data)")
    return written_paths

### Predict on specific glaciers:

In [ ]:
# import shutil

# paths_to_clear = [
#     os.path.join(cfg.dataPath, "predictions_transformer", "RGI_11_ZS"),
#     os.path.join(cfg.dataPath, "predictions_transformer", "RGI_08_ZS"),
#     os.path.join(cfg.dataPath, "predictions_transformer", "RGI_11_SR_CH"),
#     os.path.join(cfg.dataPath, "predictions_transformer", "RGI_08_SR_NOR"),
#     os.path.join(cfg.dataPath, "predictions_transformer", "RGI_11_SR_CH_LSTM"),
#     os.path.join(cfg.dataPath, "predictions_transformer", "RGI_08_SR_NOR_LSTM"),
#     *[os.path.join(cfg.dataPath, "predictions_transformer", f"RGI_11_CEU_rep{r}") for r in range(N_REPEATS)],
#     *[os.path.join(cfg.dataPath, "predictions_transformer", f"RGI_11_CH_rep{r}") for r in range(N_REPEATS)],
#     *[os.path.join(cfg.dataPath, "predictions_transformer", f"RGI_08_NOR_rep{r}") for r in range(N_REPEATS)],
# ]

# for p in paths_to_clear:
#     if os.path.exists(p):
#         shutil.rmtree(p)
#         print(f"Cleared: {p}")

In [ ]:
GLACIER_CONFIG = {
    "aletsch": {
        "rgi_ids":       monthly_cache["CH"]["data_monthly"].query("GLACIER == 'aletsch'")["RGIId"].unique().tolist(),
        "region_folder": "RGI_11_CentralEurope",
        "rgi_id_str":    "11",
        "ft_local_key":  "11_CH",
        "sr_tf_key":     "CH",
        "sr_lstm_key":   "CH",
        "df_stakes":     dfs["11"],
    },
    "rhone": {
        "rgi_ids":       monthly_cache["CH"]["data_monthly"].query("GLACIER == 'rhone'")["RGIId"].unique().tolist(),
        "region_folder": "RGI_11_CentralEurope",
        "rgi_id_str":    "11",
        "ft_local_key":  "11_CH",
        "sr_tf_key":     "CH",
        "sr_lstm_key":   "CH",
        "df_stakes":     dfs["11"],
    },
    "Nigardsbreen": {
        "rgi_ids":       monthly_cache["NOR"]["data_monthly"].query("GLACIER == 'Nigardsbreen'")["RGIId"].unique().tolist(),
        "region_folder": "RGI_08_Scandinavia",
        "rgi_id_str":    "08",
        "ft_local_key":  "08_NOR",
        "sr_tf_key":     "NOR",
        "sr_lstm_key":   "NOR",
        "df_stakes":     dfs["08"],
    },
    # "Engabreen": {
    #     "rgi_ids":       monthly_cache["NOR"]["data_monthly"].query("GLACIER == 'Engabreen'")["RGIId"].unique().tolist(),
    #     "region_folder": "RGI_08_Scandinavia",
    #     "rgi_id_str":    "08",
    #     "ft_local_key":  "08_NOR",
    #     "sr_tf_key":     "NOR",
    #     "sr_lstm_key":   "NOR",
    #     "df_stakes":     dfs["08"],
    # },
}

ft_models_for_pred = {
    "11_CH":  ft_models_ch,
    "08_NOR": ft_models_by_rgi["08"],
    "07_SJM": ft_models_by_rgi["07"],   # needs to be trained first — see below
}

In [ ]:
monthly_cache['ALA']['data_monthly'].GLACIER.unique()

In [ ]:
monthly_cache.keys()

#### Pred:

In [ ]:
PRED_BASE = os.path.join(cfg.dataPath, "predictions_transformer")

for glacier_name, gcfg in GLACIER_CONFIG.items():
    folder = gcfg["region_folder"]
    rgi_id_str = gcfg["rgi_id_str"]
    glacier_filter = gcfg["rgi_ids"]
    ds_cache_dir = os.path.join(cfg.dataPath, "RGI_v6", folder,
                                "grid_ds_cache")
    print(f"\n{'='*50}\n{glacier_name}")

    # ── Zero-shot pooled ─────────────────────────────────────────────────────
    predict_grid_from_cached_datasets(
        cfg=cfg,
        ds_cache_dir=ds_cache_dir,
        model=model_pooled,
        ds_pooled_scaler=ds_pooled_scaler,
        device=device,
        region_folder=folder,
        path_save_glw=os.path.join(PRED_BASE, f"RGI_{rgi_id_str}_ZS"),
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        glacier_filter=glacier_filter,
        period="annual",
    )

    # ── Single-region Transformer ─────────────────────────────────────────────
    sr_tf_key = gcfg["sr_tf_key"]
    if sr_tf_key in single_source_models:
        predict_grid_from_cached_datasets(
            cfg=cfg,
            ds_cache_dir=ds_cache_dir,
            model=single_source_models[sr_tf_key],
            ds_pooled_scaler=single_source_scalers[sr_tf_key],
            device=device,
            region_folder=folder,
            path_save_glw=os.path.join(PRED_BASE,
                                       f"RGI_{rgi_id_str}_SR_{sr_tf_key}"),
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
            glacier_filter=glacier_filter,
            period="annual",
        )

    # ── Single-region LSTM ────────────────────────────────────────────────────
    sr_lstm_key = gcfg["sr_lstm_key"]
    if sr_lstm_key in single_source_models_lstm:
        predict_grid_from_cached_datasets(
            cfg=cfg,
            ds_cache_dir=ds_cache_dir,
            model=single_source_models_lstm[sr_lstm_key],
            ds_pooled_scaler=single_source_scalers[sr_lstm_key],
            device=device,
            region_folder=folder,
            path_save_glw=os.path.join(
                PRED_BASE, f"RGI_{rgi_id_str}_SR_{sr_lstm_key}_LSTM"),
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
            glacier_filter=glacier_filter,
            period="annual",
        )

    # ── FT local — all reps ───────────────────────────────────────────────────
    ft_local_key = gcfg["ft_local_key"]
    if ft_local_key and ft_local_key in ft_models_for_pred:
        for rep, ft_model in ft_models_for_pred[ft_local_key]:
            predict_grid_from_cached_datasets(
                cfg=cfg,
                ds_cache_dir=ds_cache_dir,
                model=ft_model,
                ds_pooled_scaler=ds_pooled_scaler,
                device=device,
                region_folder=folder,
                path_save_glw=os.path.join(
                    PRED_BASE, f"RGI_{rgi_id_str}_FT_{ft_local_key}_rep{rep}"),
                months_head_pad=months_head_pad,
                months_tail_pad=months_tail_pad,
                glacier_filter=glacier_filter,
                period="annual",
            )

#### Alaska:

In [ ]:
# ── Alaska — pooled ZS for glaciers with stakes only ─────────────────────────
alaska_stake_glaciers = monthly_cache["ALA"]["data_monthly"]["RGIId"].unique().tolist()
print(f"Alaska stake glaciers: {len(alaska_stake_glaciers)}")

predict_grid_from_cached_datasets(
    cfg=cfg,
    ds_cache_dir=os.path.join(cfg.dataPath, "RGI_v6", "RGI_01_Alaska", "grid_ds_cache"),
    model=model_pooled,
    ds_pooled_scaler=ds_pooled_scaler,
    device=device,
    region_folder="RGI_01_Alaska",
    path_save_glw=os.path.join(PRED_BASE, "RGI_01_ZS"),
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    glacier_filter=alaska_stake_glaciers,
    period="annual",
)

## Plots:

### Stake locations:

In [ ]:
def plot_pred_vs_obs_stakes(
    glaciers,  # {glacier_name: (rgi_id, df_stakes)}
    models,  # {model_label: {glacier_name: str | list[str] | None}}
    years,
    period="annual",
    apply_smoothing=True,
    text_size=10,
    cm=1 / 2.54,
):
    """
    Pred vs obs scatter at stake locations, all years pooled.
    One row per glacier, one column per model.
    If a list of paths is given for a model, predictions are averaged across
    reps and error bars show ±1 std across reps.
    """
    from sklearn.metrics import root_mean_squared_error
    from scipy.stats import pearsonr

    n_glaciers = len(glaciers)
    n_models = len(models)

    fig, axes = plt.subplots(
        n_glaciers,
        n_models,
        figsize=(n_models * 10 * cm, n_glaciers * 10 * cm),
        constrained_layout=True,
        dpi=200,
        squeeze=False,
    )

    # pre-compute per-glacier shared axis limits across all models
    row_lims = {}
    for glacier_name, (rgi_id, df_stakes) in glaciers.items():
        all_vals = []
        for model_label, paths_by_glacier in models.items():
            paths = paths_by_glacier.get(glacier_name)
            if paths is None:
                continue
            if isinstance(paths, str):
                paths = [paths]
            for year in years:
                sub = df_stakes[(df_stakes["GLACIER"] == glacier_name)
                                & (df_stakes["YEAR"] == year) &
                                (df_stakes["PERIOD"] == period)]
                if sub.empty:
                    continue
                all_vals.extend(sub["POINT_BALANCE"].tolist())
                for path in paths:
                    ds = _load_lstm_ds(
                        glacier=rgi_id,
                        year=year,
                        path_pred_lstm=path,
                        period=period,
                        apply_smoothing_fn=apply_gaussian_filter
                        if apply_smoothing else None,
                    )
                    if ds is None or "pred_masked" not in ds:
                        continue

                    def _sample(row):
                        try:
                            return ds.sel(
                                lat=row["POINT_LAT"],
                                lon=row["POINT_LON"],
                                method="nearest")["pred_masked"].item()
                        except Exception:
                            return np.nan

                    preds = sub.apply(_sample, axis=1).dropna().tolist()
                    all_vals.extend(preds)

        if all_vals:
            margin = 0.2
            row_lims[glacier_name] = [
                min(all_vals) - margin,
                max(all_vals) + margin
            ]
        else:
            row_lims[glacier_name] = [-12, 5]

    for row_idx, (glacier_name, (rgi_id,
                                 df_stakes)) in enumerate(glaciers.items()):
        for col_idx, (model_label,
                      paths_by_glacier) in enumerate(models.items()):
            ax = axes[row_idx, col_idx]

            paths = paths_by_glacier.get(glacier_name)
            if paths is None:
                ax.set_title(f"{glacier_name} / {model_label} — N/A")
                ax.set_axis_off()
                continue

            # normalise to list
            if isinstance(paths, str):
                paths = [paths]

            all_obs, all_pred_mean, all_pred_std, all_years = [], [], [], []

            for year in years:
                sub = df_stakes[(df_stakes["GLACIER"] == glacier_name)
                                & (df_stakes["YEAR"] == year) &
                                (df_stakes["PERIOD"] == period)].copy()
                if sub.empty:
                    continue

                # sample each rep
                rep_preds = []
                for path in paths:
                    ds = _load_lstm_ds(
                        glacier=rgi_id,
                        year=year,
                        path_pred_lstm=path,
                        period=period,
                        apply_smoothing_fn=apply_gaussian_filter
                        if apply_smoothing else None,
                    )
                    if ds is None or "pred_masked" not in ds:
                        continue

                    def _sample(row):
                        try:
                            return ds.sel(
                                lat=row["POINT_LAT"],
                                lon=row["POINT_LON"],
                                method="nearest")["pred_masked"].item()
                        except Exception:
                            return np.nan

                    rep_preds.append(sub.apply(_sample, axis=1).values)

                if not rep_preds:
                    continue

                rep_array = np.stack(rep_preds, axis=0)  # (n_reps, n_stakes)
                pred_mean = np.nanmean(rep_array, axis=0)
                pred_std = np.nanstd(rep_array, axis=0)  # 0 if single rep

                sub["pred_mean"] = pred_mean
                sub["pred_std"] = pred_std
                sub = sub.dropna(subset=["pred_mean", "POINT_BALANCE"])

                all_obs.extend(sub["POINT_BALANCE"].tolist())
                all_pred_mean.extend(sub["pred_mean"].tolist())
                all_pred_std.extend(sub["pred_std"].tolist())
                all_years.extend([year] * len(sub))

            if not all_obs:
                ax.set_title(
                    f"{glacier_name.capitalize()} / {model_label} — no data")
                ax.set_axis_off()
                continue

            obs = np.array(all_obs)
            pred_mean = np.array(all_pred_mean)
            pred_std = np.array(all_pred_std)
            years_arr = np.array(all_years)

            rmse = root_mean_squared_error(obs, pred_mean)
            r, _ = pearsonr(obs, pred_mean)
            bias = float(np.mean(pred_mean - obs))

            # scatter coloured by year
            sc = ax.scatter(obs,
                            pred_mean,
                            c=years_arr,
                            cmap="viridis",
                            s=15,
                            alpha=0.8,
                            zorder=3)
            plt.colorbar(sc, ax=ax, label="Year")

            # error bars only if we have multiple reps
            if len(paths) > 1:
                ax.errorbar(
                    obs,
                    pred_mean,
                    yerr=pred_std,
                    fmt="none",
                    ecolor="grey",
                    alpha=0.4,
                    elinewidth=0.8,
                    capsize=2,
                    zorder=2,
                )

            lim = row_lims[glacier_name]
            ax.plot(lim, lim, "k--", lw=0.8)
            ax.set_xlim(lim)
            ax.set_ylim(lim)

            ax.set_xlabel("Observed MB [m w.e.]", fontsize=text_size)
            ax.set_ylabel("Predicted MB [m w.e.]", fontsize=text_size)
            ax.set_title(
                f"{glacier_name.capitalize()}\n {model_label} ({len(obs)} stakes)",
                fontsize=text_size + 1,
            )
            n_reps_str = f", {len(paths)} reps" if len(paths) > 1 else ""
            ax.text(
                0.05,
                0.95,
                f"RMSE: {rmse:.2f}\nr: {r:.2f}\nbias: {bias:.2f}{n_reps_str}",
                transform=ax.transAxes,
                fontsize=text_size - 1,
                va="top",
                ha="left",
                bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"),
            )
            ax.grid(alpha=0.2)

    return fig

In [ ]:
fig = plot_pred_vs_obs_stakes(
    glaciers={
        name: (gcfg["rgi_ids"][0], gcfg["df_stakes"])
        for name, gcfg in GLACIER_CONFIG.items()
    },
    models={
        "Zero-Shot (LSTM)": {
            name: os.path.join(PRED_BASE, f"RGI_{gcfg['rgi_id_str']}_SR_{gcfg['sr_lstm_key']}_LSTM")
            if gcfg["sr_lstm_key"] in single_source_models_lstm else None
            for name, gcfg in GLACIER_CONFIG.items()
        },
        "Zero-Shot (Transformer)": {
            name: os.path.join(PRED_BASE, f"RGI_{gcfg['rgi_id_str']}_SR_{gcfg['sr_tf_key']}")
            if gcfg["sr_tf_key"] in single_source_models else None
            for name, gcfg in GLACIER_CONFIG.items()
        },
        # "Multi-region ZS": {
        #     name: os.path.join(PRED_BASE, f"RGI_{gcfg['rgi_id_str']}_ZS")
        #     for name, gcfg in GLACIER_CONFIG.items()
        # },
        # "FT locally": {
        #     name: [
        #         os.path.join(PRED_BASE, f"RGI_{gcfg['rgi_id_str']}_FT_{gcfg['ft_local_key']}_rep{rep}")
        #         for rep, _ in ft_models_for_pred.get(gcfg["ft_local_key"], [])
        #     ] if gcfg["ft_local_key"] else None
        #     for name, gcfg in GLACIER_CONFIG.items()
        # },
    },
    years=list(range(2005, 2025)),
    period="annual",
)

#### Gradients:

In [ ]:
def aggregate_gridded_transformer_by_glacier(rgi_id, path_pred, period="annual"):
    lstm_tag = "annual" if period in ("annual", "ann", "yearly") else "winter"
    base = os.path.join(path_pred, rgi_id)

    years = []
    if os.path.isdir(base):
        for f in glob.glob(os.path.join(base, f"*_{lstm_tag}.zarr")):
            m = re.match(r".*[\\/].+?_(\d{4})_" + re.escape(lstm_tag) + r"\.zarr$", f)
            if m:
                years.append(int(m.group(1)))
    years = sorted(set(years))

    if not years:
        print(f"  No predictions found for {rgi_id} in {base}")
        return pd.DataFrame()

    dfs = []
    for year in years:
        zarr_path = os.path.join(base, f"{rgi_id}_{year}_{lstm_tag}.zarr")
        if not os.path.exists(zarr_path):
            continue
        try:
            ds = apply_gaussian_filter(xr.open_dataset(zarr_path, engine="zarr"))
            lon_name = "lon" if "lon" in ds.coords else "longitude"
            lat_name = "lat" if "lat" in ds.coords else "latitude"

            df = (
                ds["pred_masked"].to_dataframe().reset_index()
                .drop(["x", "y"], axis=1, errors="ignore")
                .merge(
                    ds["masked_elev"].to_dataframe().reset_index()
                    .drop(["x", "y"], axis=1, errors="ignore"),
                    on=[lat_name, lon_name], how="left",
                )
                .dropna()
                .rename(columns={
                    "pred_masked": "pred",
                    "masked_elev": "POINT_ELEVATION",
                    lat_name: "lat", lon_name: "lon",
                })
            )
            df["YEAR"] = year
            df["PERIOD"] = lstm_tag

            min_alt = np.floor(df["POINT_ELEVATION"].min() / 100) * 100
            max_alt = np.ceil(df["POINT_ELEVATION"].max() / 100) * 100
            bins = np.arange(min_alt, max_alt + 100, 100)
            df["altitude_interval"] = pd.cut(df["POINT_ELEVATION"], bins=bins, right=False)
            centers = {iv: round((iv.left + iv.right) / 2)
                       for iv in df["altitude_interval"].cat.categories}
            df["altitude_interval"] = df["altitude_interval"].map(centers)
            dfs.append(df)
        except Exception as e:
            print(f"  [error] {rgi_id} {year}: {e}")

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

In [ ]:
def plot_transformer_gradients(
    GLACIER_CONFIG,
    PRED_BASE,
    models,
    period="annual",
    ncols=None,
    figsize_y=9,
    cm=1 / 2.54,
    text_size=8,
    colors=None,
):
    glaciers = list(GLACIER_CONFIG.items())
    n = len(glaciers)
    ncols = ncols or n
    nrows = math.ceil(n / ncols)

    if colors is None:
        prop_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
        colors = {label: prop_cycle[i % len(prop_cycle)] for i, label in enumerate(models)}

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols * figsize_y * cm, nrows * 12 * cm),
        constrained_layout=True, dpi=200,
    )
    axes = np.array(axes).reshape(nrows, ncols)

    for idx, (glacier_name, gcfg) in enumerate(glaciers):
        r, c = idx // ncols, idx % ncols
        ax = axes[r, c]
        rgi_id = gcfg["rgi_ids"][0]
        df_stakes = gcfg["df_stakes"]   # ← per-glacier stakes from GLACIER_CONFIG

        for label, path_template in models.items():
            path = os.path.join(PRED_BASE, path_template.format(**gcfg))
            df = aggregate_gridded_transformer_by_glacier(rgi_id, path, period=period)
            if df.empty:
                continue

            grouped = df.groupby("altitude_interval")["pred"]
            mean = grouped.mean()
            std  = grouped.std().fillna(0)

            ax.plot(mean.values, mean.index, color=colors[label], lw=1.2, label=label)
            ax.fill_betweenx(
                mean.index,
                mean.values - std.values,
                mean.values + std.values,
                alpha=0.15, color=colors[label],
            )

        # stake overlay — use glacier's own df_stakes
        if df_stakes is not None:
            sub = df_stakes[
                (df_stakes["GLACIER"] == glacier_name) &
                (df_stakes["PERIOD"] == period)
            ].copy()
            if not sub.empty and "POINT_ELEVATION" in sub.columns:
                bins = np.arange(
                    np.floor(sub["POINT_ELEVATION"].min() / 20) * 20,
                    np.ceil(sub["POINT_ELEVATION"].max() / 20) * 20 + 20,
                    20,
                )
                sub["alt_bin"] = pd.cut(sub["POINT_ELEVATION"], bins=bins, right=False)
                centers = {iv: round((iv.left + iv.right) / 2)
                           for iv in sub["alt_bin"].cat.categories}
                sub["alt_bin"] = sub["alt_bin"].map(centers)
                stake_mean = sub.groupby("alt_bin")["POINT_BALANCE"].mean()
                ax.scatter(
                    stake_mean.values, stake_mean.index,
                    marker="o", s=18, facecolors="none",
                    edgecolors="black", lw=0.8, label="Stakes", zorder=5,
                )

        ax.axvline(0, color="k", lw=0.6, ls="--", alpha=0.5)
        ax.set_title(glacier_name.capitalize(), fontsize=text_size + 1)
        ax.set_xlabel("MB [m w.e.]", fontsize=text_size)
        ax.set_ylabel("Elevation (m a.s.l.)" if c == 0 else "", fontsize=text_size)
        ax.grid(alpha=0.2)
        ax.tick_params(labelsize=text_size - 1)
        if idx == 0:
            ax.legend(fontsize=text_size - 1)

    for j in range(idx + 1, nrows * ncols):
        axes[j // ncols, j % ncols].set_axis_off()

    return fig

In [ ]:
fig = plot_transformer_gradients(
    GLACIER_CONFIG=GLACIER_CONFIG,
    PRED_BASE=PRED_BASE,
    models={
        "SR ZS (Transformer)": "RGI_{rgi_id_str}_SR_{sr_tf_key}",
        "SR ZS (LSTM)":        "RGI_{rgi_id_str}_SR_{sr_lstm_key}_LSTM",
    },
    period="annual",
)

In [ ]:
rgi_id = GLACIER_CONFIG["Nigardsbreen"]["rgi_ids"][0]
path_sr_tf = os.path.join(PRED_BASE, f"RGI_08_SR_NOR")

df = aggregate_gridded_transformer_by_glacier(rgi_id, path_sr_tf, period="annual")

# look at spread within the lowest elevation bin
low = df[df["POINT_ELEVATION"] < 600]
print(f"N pixels below 600m: {len(low)}")
print(f"Mean pred: {low['pred'].mean():.2f}, std: {low['pred'].std():.2f}")
print(f"Min pred: {low['pred'].min():.2f}, Max: {low['pred'].max():.2f}")

# and check how many unique lat/lon positions
print(f"Unique locations: {low[['lat','lon']].drop_duplicates().shape[0]}")

### Maps:

In [ ]:
def _stake_overlay_transformer(ax, glacier, year, cmap, norm, ds_pred,
                               df_stakes, period):
    """Overlay stakes on transformer prediction map, return RMSE if possible."""
    if df_stakes is None or ds_pred is None:
        return None

    sub = df_stakes[(df_stakes["GLACIER"] == glacier)
                    & (df_stakes["YEAR"] == year) &
                    (df_stakes["PERIOD"] == period)].copy()

    if sub.empty:
        return None

    def _sample(row):
        try:
            return ds_pred.sel(lat=row["POINT_LAT"],
                               lon=row["POINT_LON"],
                               method="nearest")["pred_masked"].item()
        except Exception:
            return np.nan

    sub["pred_at_stake"] = sub.apply(_sample, axis=1)
    sub = sub.dropna(subset=["pred_at_stake"])

    if not sub.empty:
        sns.scatterplot(
            data=sub,
            x="POINT_LON",
            y="POINT_LAT",
            hue="POINT_BALANCE",
            palette=cmap,
            hue_norm=norm,
            ax=ax,
            s=25,
            legend=False,
        )

    if "POINT_BALANCE" in sub.columns and len(sub) > 0:
        return root_mean_squared_error(sub["POINT_BALANCE"],
                                       sub["pred_at_stake"])
    return None


def plot_transformer_grids(
    glacier_name,
    rgi_id,
    years,
    path_pred_transformer,  # path_save_glw used in predict_grid_from_cached_datasets
    df_stakes=None,
    period="annual",
    apply_smoothing=True,
    ncols=4,
    text_size=8,
    figsize_y=9,
    cm=1 / 2.54,
):
    datasets, year_list = [], []

    for year in years:
        ds = _load_lstm_ds(
            glacier=rgi_id,
            year=year,
            path_pred_lstm=path_pred_transformer,
            period=period,
            apply_smoothing_fn=apply_gaussian_filter
            if apply_smoothing else None,
        )
        if ds is None or "pred_masked" not in ds:
            continue
        datasets.append(ds)
        year_list.append(year)

    if not datasets:
        raise RuntimeError(
            f"No predictions found for {rgi_id} in {path_pred_transformer}")

    global_vmin = min(float(ds["pred_masked"].min()) for ds in datasets)
    global_vmax = max(float(ds["pred_masked"].max()) for ds in datasets)
    cmap, norm = get_color_maps(global_vmin, global_vmax)
    vmin_display = max(global_vmin, -12)
    vmax_display = min(global_vmax, 5)

    n = len(datasets)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * figsize_y * cm, nrows * 8 * cm),
        constrained_layout=True,
        dpi=200,
    )
    if nrows == 1:
        axes = axes.reshape(1, -1)

    mappable_for_cb = None
    label_iter = iter(string.ascii_lowercase)

    for idx, (ds, year) in enumerate(zip(datasets, year_list)):
        r, c = idx // ncols, idx % ncols
        ax = axes[r, c]

        mappable_for_cb = ds["pred_masked"].plot.imshow(ax=ax,
                                                        cmap=cmap,
                                                        norm=norm,
                                                        add_colorbar=False)
        ax.set_aspect("auto")
        ax.set_title(f"{glacier_name.capitalize()} – Transformer ({year})",
                     fontsize=text_size + 1)

        rmse = _stake_overlay_transformer(
            ax=ax,
            glacier=glacier_name,
            year=year,
            cmap=cmap,
            norm=norm,
            ds_pred=ds,
            df_stakes=df_stakes,
            period=period,
        )

        mean_m = float(ds["pred_masked"].mean())
        annotation = (f"RMSE: {rmse:.2f}\n"
                      if rmse else "") + f"mean MB: {mean_m:.2f}"
        ax.text(0.04,
                0.03,
                annotation,
                transform=ax.transAxes,
                fontsize=text_size,
                ha="left",
                va="bottom")
        ax.text(0.03,
                0.97,
                f"({next(label_iter)})",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=text_size)
        ax.grid(alpha=0.2)
        ax.tick_params(axis="both", labelsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("")

    for j in range(idx + 1, nrows * ncols):
        axes[j // ncols, j % ncols].set_axis_off()

    fig_cb = plt.figure(figsize=(1.5 * cm, 12 * cm), dpi=200)
    ax_cb = fig_cb.add_axes([0.25, 0.05, 0.35, 0.90])
    cb = fig_cb.colorbar(mappable_for_cb, cax=ax_cb)
    cb.ax.set_ylim(vmin_display, vmax_display)
    if global_vmin < vmin_display:
        ax_cb.plot([0.5], [-0.02],
                   marker="v",
                   color="black",
                   markersize=4,
                   transform=ax_cb.transAxes,
                   clip_on=False)
    if global_vmax > vmax_display:
        ax_cb.plot([0.5], [1.02],
                   marker="^",
                   color="black",
                   markersize=4,
                   transform=ax_cb.transAxes,
                   clip_on=False)
    cb.set_label("Mass Balance [m w.e.]")

    return fig, fig_cb

In [ ]:
fig, fig_cb = plot_transformer_grids(
    glacier_name="aletsch",
    rgi_id=aletsch_ids[0],
    years=list(range(2005, 2025)),
    path_pred_transformer=os.path.join(cfg.dataPath, "predictions_transformer",
                                       "RGI_11_CEU"),
    df_stakes=dfs["11"],
    period="annual",
)

fig, fig_cb = plot_transformer_grids(
    glacier_name="Nigardsbreen",
    rgi_id=nigards_ids[0],
    years=list(range(2005, 2025)),
    path_pred_transformer=os.path.join(cfg.dataPath, "predictions_transformer",
                                       "RGI_08_NOR"),
    df_stakes=dfs["08"],
    period="annual",
)

#### Predict on all RGIs:

In [ ]:
_, ft_model_ceu = ft_assets_ceu[0]["rep"], ft_assets_ceu[0]["model"]

predict_grid_from_cached_datasets(
    cfg=cfg,
    ds_cache_dir=os.path.join(cfg.dataPath, "RGI_v6", "RGI_11_CentralEurope",
                              "grid_ds_cache"),
    model=ft_model_ceu,
    ds_pooled_scaler=ds_pooled_scaler,
    device=device,
    region_folder="RGI_11_CentralEurope",
    path_save_glw=os.path.join(cfg.dataPath, "predictions_transformer",
                               "RGI_11_CEU"),
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    period="annual",
)

In [ ]:
RGI_REGION_FOLDERS = {
    "06": ("RGI_06_Iceland", "ISL"),
    "07": ("RGI_07_Svalbard", "SJM"),
    "08": ("RGI_08_Scandinavia", "NOR"),
}

for rgi_id, (folder, ft_code) in RGI_REGION_FOLDERS.items():
    _, ft_model = ft_models_by_rgi[rgi_id][0]  # rep 0

    predict_grid_from_cached_datasets(
        cfg=cfg,
        ds_cache_dir=os.path.join(cfg.dataPath, "RGI_v6", folder,
                                  "grid_ds_cache"),
        model=ft_model,
        ds_pooled_scaler=ds_pooled_scaler,
        device=device,
        region_folder=folder,
        path_save_glw=os.path.join(cfg.dataPath, "predictions_transformer",
                                   f"RGI_{rgi_id}_{ft_code}"),
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        period="annual",
    )